# Chlorophyll Fluorescence in Ocean Color Remote Sensing

## This notebook demonstrates and tests the chlorophyll fluorescence calculations in the BING radiative transfer module.

Chlorophyll fluorescence occurs when light absorbed by photosynthetic pigments in phytoplankton is re-emitted at longer wavelengths. The primary emission peak is centered at **685 nm** (PS II), with a secondary peak around **730 nm** (PS I).

Key characteristics:
- **Excitation range**: 370-690 nm (absorbed by photosynthetic pigments)
- **Primary emission**: 685 nm (FWHM ~25 nm)
- **Secondary emission**: ~730 nm (FWHM ~50 nm)
- **Quantum yield**: 0.005-0.07 (varies with irradiance and physiology)

**References:**
- Gordon, H.R. (1979). Appl. Opt. 18, 1161-1166
- Maritorena et al. (2000). Appl. Opt. 39, 6725-6737
- Ocean Optics Web Book: Chlorophyll Fluorescence

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bing.rt import chl_fl
from bing.rt import raman
from bing.tests import test_chl_fl

%matplotlib inline

## 1. Emission Line Shape

### 1.1 Single Gaussian Model

The simplest model represents the fluorescence emission as a single Gaussian centered at 685 nm with FWHM = 25 nm.

In [ ]:
# Generate emission spectrum
wavelengths = np.linspace(600, 850, 500)

# Single Gaussian emission line
h_single = chl_fl.emission_line_single_gaussian(wavelengths)

# Double Gaussian emission line (includes PS I peak at 730 nm)
h_double = chl_fl.emission_line_double_gaussian(wavelengths)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(wavelengths, h_single, 'b-', linewidth=2, label='Single Gaussian (685 nm)')
ax.plot(wavelengths, h_double, 'r--', linewidth=2, label='Double Gaussian (685 + 730 nm)')
ax.axvline(x=685, color='b', linestyle=':', alpha=0.5)
ax.axvline(x=730, color='r', linestyle=':', alpha=0.5)
ax.set_xlabel('Emission Wavelength (nm)', fontsize=12)
ax.set_ylabel('h$_C$(λ) (nm$^{-1}$)', fontsize=12)
ax.set_title('Chlorophyll Fluorescence Emission Line Shape', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(600, 850)
plt.tight_layout()
plt.show()

# Verify normalization
integral_single = np.trapz(h_single, wavelengths)
integral_double = np.trapz(h_double, wavelengths)
print(f"Single Gaussian integral: {integral_single:.4f} (should be ~1.0)")
print(f"Double Gaussian integral: {integral_double:.4f} (should be ~1.0)")

### 1.2 Emission Line Parameters

In [ ]:
print("Chlorophyll Fluorescence Emission Parameters")
print("=" * 50)
print(f"\nPrimary peak (PS II):")
print(f"  Wavelength: {chl_fl.LAMBDA_FL_PRIMARY} nm")
print(f"  FWHM: {chl_fl.FWHM_FL_PRIMARY} nm")
print(f"  Sigma: {chl_fl.SIGMA_FL_PRIMARY:.1f} nm")
print(f"\nSecondary peak (PS I):")
print(f"  Wavelength: {chl_fl.LAMBDA_FL_SECONDARY} nm")
print(f"  FWHM: {chl_fl.FWHM_FL_SECONDARY} nm")
print(f"  Sigma: {chl_fl.SIGMA_FL_SECONDARY:.1f} nm")
print(f"\nDouble Gaussian weights:")
print(f"  Primary: {chl_fl.WEIGHT_PRIMARY}")
print(f"  Secondary: {chl_fl.WEIGHT_SECONDARY}")
print(f"\nExcitation range: {chl_fl.LAMBDA_EX_MIN}-{chl_fl.LAMBDA_EX_MAX} nm")

## 2. Quantum Yield

### 2.1 Quantum Yield vs Irradiance

Fluorescence quantum yield (Φ$_C$) varies significantly with light conditions due to nonphotochemical quenching (NPQ):
- **High irradiance (surface)**: Φ$_C$ = 0.005-0.01
- **Low irradiance (depth)**: Φ$_C$ up to 0.07

In [ ]:
# PAR range (μmol photons m^-2 s^-1)
PAR = np.linspace(0, 2000, 200)

# Calculate quantum yield
phi_C = chl_fl.quantum_yield_irradiance_dependent(PAR)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(PAR, phi_C * 100, 'g-', linewidth=2)
ax.axhline(y=chl_fl.PHI_FL_DEFAULT * 100, color='r', linestyle='--', 
           label=f'HydroLight default = {chl_fl.PHI_FL_DEFAULT}')
ax.axhline(y=chl_fl.PHI_FL_HIGH_LIGHT * 100, color='orange', linestyle=':', 
           label=f'High light limit = {chl_fl.PHI_FL_HIGH_LIGHT}')
ax.axhline(y=chl_fl.PHI_FL_LOW_LIGHT * 100, color='b', linestyle=':', 
           label=f'Low light limit = {chl_fl.PHI_FL_LOW_LIGHT}')
ax.set_xlabel('PAR (μmol photons m$^{-2}$ s$^{-1}$)', fontsize=12)
ax.set_ylabel('Quantum Yield Φ$_C$ (%)', fontsize=12)
ax.set_title('Fluorescence Quantum Yield vs Irradiance', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2000)
ax.set_ylim(0, 8)
plt.tight_layout()
plt.show()

### 2.2 Quantum Yield vs Depth

Combining light attenuation with the irradiance-dependent quantum yield gives a depth profile.

In [ ]:
# Depth range
depths = np.linspace(0, 100, 100)

# Different K_PAR values (diffuse attenuation)
K_values = [0.03, 0.05, 0.1, 0.2]  # m^-1
colors = ['blue', 'green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(8, 8))

for K, color in zip(K_values, colors):
    phi_z = chl_fl.quantum_yield_depth_profile(depths, K_PAR=K)
    ax.plot(phi_z * 100, depths, color=color, linewidth=2, label=f'K$_{{PAR}}$ = {K} m$^{{-1}}$')

ax.invert_yaxis()
ax.set_xlabel('Quantum Yield Φ$_C$ (%)', fontsize=12)
ax.set_ylabel('Depth (m)', fontsize=12)
ax.set_title('Fluorescence Quantum Yield Depth Profile', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Fluorescence Coefficients

### 3.1 Scattering and Backscattering Coefficients

The fluorescence "scattering" coefficient is:
$$b_C(\lambda') = \Phi_C \times a_{ph}(\lambda')$$

For isotropic emission, the backscattering coefficient is:
$$b_{bC}(\lambda') = 0.5 \times b_C(\lambda')$$

In [ ]:
# Phytoplankton absorption at different wavelengths (typical values)
wavelengths_ex = np.array([400, 420, 440, 460, 480, 500, 550, 600, 650])
a_ph = np.array([0.02, 0.025, 0.03, 0.025, 0.02, 0.015, 0.01, 0.008, 0.015])  # m^-1

phi_C = 0.02  # Default quantum yield

# Calculate coefficients
b_C = chl_fl.fluorescence_scattering_coeff(a_ph, phi_C)
bb_C = chl_fl.fluorescence_backscattering_coeff(a_ph, phi_C)

print("Fluorescence Coefficients")
print("=" * 60)
print(f"{'λ (nm)':<10} {'a_ph (m⁻¹)':<15} {'b_C (m⁻¹)':<15} {'b_bC (m⁻¹)':<15}")
print("-" * 60)
for wl, a, b, bb in zip(wavelengths_ex, a_ph, b_C, bb_C):
    print(f"{wl:<10} {a:<15.4f} {b:<15.2e} {bb:<15.2e}")

# Compare with Raman backscattering
bb_R = raman.raman_backscattering_coeff(440)
print(f"\nComparison at 440 nm:")
print(f"  Fluorescence b_bC = {bb_C[2]:.2e} m⁻¹")
print(f"  Raman b_bR = {bb_R:.2e} m⁻¹")
print(f"  Ratio b_bC/b_bR = {bb_C[2]/bb_R:.2f}")

### 3.2 Comparison with Raman Scattering

In [ ]:
# Wavelength range for excitation
wave_ex = np.linspace(400, 600, 100)

# Approximate phytoplankton absorption spectrum (simplified)
a_ph_spectrum = 0.03 * np.exp(-((wave_ex - 440) / 40)**2)

# Fluorescence backscattering (varies with a_ph)
bb_fl = chl_fl.fluorescence_backscattering_coeff(a_ph_spectrum, phi_C=0.02)

# Raman backscattering (varies with wavelength)
bb_raman = raman.raman_backscattering_coeff(wave_ex)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(wave_ex, bb_fl * 1e4, 'g-', linewidth=2, label='Fluorescence b$_{bC}$')
ax.semilogy(wave_ex, bb_raman * 1e4, 'b-', linewidth=2, label='Raman b$_{bR}$')
ax.set_xlabel('Excitation Wavelength (nm)', fontsize=12)
ax.set_ylabel('Backscattering Coefficient (×10$^{-4}$ m$^{-1}$)', fontsize=12)
ax.set_title('Fluorescence vs Raman Backscattering', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(400, 600)
plt.tight_layout()
plt.show()

## 4. Wavelength Redistribution Function

The redistribution function $f_C(\lambda', \lambda)$ describes the probability that light absorbed at excitation wavelength $\lambda'$ will be emitted at wavelength $\lambda$:

$$f_C(\lambda', \lambda) = \Phi_C \times g_C(\lambda') \times h_C(\lambda) \times \frac{\lambda'}{\lambda}$$

In [ ]:
# Emission wavelength range
wave_em = np.linspace(640, 800, 200)

# Different excitation wavelengths
excitation_wavelengths = [400, 440, 500, 550, 600]
colors = plt.cm.viridis(np.linspace(0, 1, len(excitation_wavelengths)))

fig, ax = plt.subplots(figsize=(10, 6))

for lambda_ex, color in zip(excitation_wavelengths, colors):
    f_C = chl_fl.wavelength_redistribution(lambda_ex, wave_em, phi_C=0.02)
    # Normalize for plotting
    f_norm = f_C / np.max(f_C) if np.max(f_C) > 0 else f_C
    ax.plot(wave_em, f_norm, color=color, linewidth=2, label=f"λ' = {lambda_ex} nm")

ax.axvline(x=685, color='gray', linestyle='--', alpha=0.5, label='685 nm peak')
ax.set_xlabel('Emission Wavelength λ (nm)', fontsize=12)
ax.set_ylabel('Normalized f$_C$(λ\', λ)', fontsize=12)
ax.set_title('Fluorescence Wavelength Redistribution', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(640, 800)
plt.tight_layout()
plt.show()

## 5. Fluorescence Reflectance Contribution

### 5.1 Single Wavelength Calculation

In [ ]:
# Typical conditions at 685 nm (emission) with 440 nm excitation
# At emission wavelength (685 nm) - high water absorption
a_w_em = 0.45      # Pure water absorption
a_ph_em = 0.01     # Low phytoplankton absorption in red
bb_w_em = 0.0005   # Pure water backscattering
bb_p_em = 0.001    # Particle backscattering

a_em = a_w_em + a_ph_em
bb_em = bb_w_em + bb_p_em

# At excitation wavelength (440 nm) - blue absorption peak
a_w_ex = 0.01      # Pure water absorption
a_ph_ex = 0.03     # Phytoplankton absorption
bb_w_ex = 0.003    # Pure water backscattering
bb_p_ex = 0.002    # Particle backscattering

a_ex = a_w_ex + a_ph_ex
bb_ex = bb_w_ex + bb_p_ex

# Calculate fluorescence reflectance
R_F = chl_fl.calc_R_fluorescence(
    a_em, bb_em, a_ex, bb_ex, a_ph_ex, phi_C=0.02
)

print("Fluorescence Reflectance Calculation")
print("=" * 50)
print(f"\nIOPs at emission wavelength (685 nm):")
print(f"  Total absorption a = {a_em:.3f} m⁻¹")
print(f"  Total backscatter bb = {bb_em:.4f} m⁻¹")
print(f"\nIOPs at excitation wavelength (440 nm):")
print(f"  Total absorption a = {a_ex:.3f} m⁻¹")
print(f"  Total backscatter bb = {bb_ex:.4f} m⁻¹")
print(f"  Phytoplankton absorption a_ph = {a_ph_ex:.3f} m⁻¹")
print(f"\nQuantum yield Φ_C = 0.02")
print(f"\nFluorescence reflectance R_F = {R_F:.4e}")

### 5.2 Sensitivity to Chlorophyll Concentration

In [ ]:
# Vary phytoplankton absorption (proxy for chlorophyll)
a_ph_range = np.linspace(0.005, 0.2, 50)  # m^-1

# Calculate fluorescence reflectance for each a_ph value
R_F_values = []
for a_ph in a_ph_range:
    a_ex_total = 0.01 + a_ph
    R_F = chl_fl.calc_R_fluorescence(
        a_em=0.46, bb_em=0.0015,
        a_ex=a_ex_total, bb_ex=0.005,
        a_ph_ex=a_ph, phi_C=0.02
    )
    R_F_values.append(R_F)

R_F_values = np.array(R_F_values)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(a_ph_range, R_F_values * 1e4, 'g-', linewidth=2)
ax.set_xlabel('Phytoplankton Absorption a$_{ph}$(440) (m$^{-1}$)', fontsize=12)
ax.set_ylabel('Fluorescence Reflectance R$_F$ (×10$^{-4}$)', fontsize=12)
ax.set_title('Fluorescence Reflectance vs Chlorophyll', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Sensitivity to Quantum Yield

In [ ]:
# Vary quantum yield
phi_range = np.linspace(0.005, 0.07, 50)

# Calculate fluorescence reflectance
R_F_phi = []
for phi in phi_range:
    R_F = chl_fl.calc_R_fluorescence(
        a_em=0.46, bb_em=0.0015,
        a_ex=0.04, bb_ex=0.005,
        a_ph_ex=0.03, phi_C=phi
    )
    R_F_phi.append(R_F)

R_F_phi = np.array(R_F_phi)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(phi_range * 100, R_F_phi * 1e4, 'b-', linewidth=2)
ax.axvline(x=2, color='r', linestyle='--', label='Default Φ$_C$ = 0.02')
ax.set_xlabel('Quantum Yield Φ$_C$ (%)', fontsize=12)
ax.set_ylabel('Fluorescence Reflectance R$_F$ (×10$^{-4}$)', fontsize=12)
ax.set_title('Fluorescence Reflectance vs Quantum Yield', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Fluorescence Line Height (FLH)

FLH is a standard satellite product that isolates the fluorescence signal from the background reflectance using three wavelengths around 680 nm.

$$\text{FLH} = R_{rs}(680) - R_{rs,baseline}(680)$$

where the baseline is linearly interpolated between ~665 nm and ~709 nm.

In [ ]:
# Example Rrs values (with fluorescence peak)
Rrs_665 = 0.001
Rrs_680 = 0.0015  # Higher due to fluorescence
Rrs_709 = 0.0008

# Calculate FLH
FLH = chl_fl.calc_fluorescence_line_height(Rrs_665, Rrs_680, Rrs_709)
nFLH = chl_fl.calc_normalized_fluorescence_line_height(Rrs_665, Rrs_680, Rrs_709)

# Calculate baseline for visualization
wavelengths_flh = np.array([665, 680, 709])
Rrs_vals = np.array([Rrs_665, Rrs_680, Rrs_709])
w = (680 - 665) / (709 - 665)
baseline_680 = Rrs_665 + w * (Rrs_709 - Rrs_665)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(wavelengths_flh, Rrs_vals * 1e3, 'bo-', markersize=10, linewidth=2, label='Rrs')
ax.plot([665, 709], [Rrs_665 * 1e3, Rrs_709 * 1e3], 'r--', linewidth=2, label='Baseline')
ax.plot([680, 680], [baseline_680 * 1e3, Rrs_680 * 1e3], 'g-', linewidth=3, label=f'FLH = {FLH*1e3:.3f} ×10⁻³ sr⁻¹')
ax.scatter([680], [baseline_680 * 1e3], color='r', s=100, zorder=5)
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Rrs (×10$^{-3}$ sr$^{-1}$)', fontsize=12)
ax.set_title('Fluorescence Line Height (FLH)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(655, 720)
plt.tight_layout()
plt.show()

print(f"FLH = {FLH:.4e} sr⁻¹")
print(f"Normalized FLH = {nFLH:.4f}")

## 7. Summary Function

In [ ]:
# Get summary at typical excitation wavelength
summary = chl_fl.summary_at_wavelength(
    wavelength_ex=440.0,
    a_ph=0.03,
    phi_C=0.02
)

print("Fluorescence Summary at λ' = 440 nm")
print("=" * 50)
for key, value in summary.items():
    if isinstance(value, float):
        if value < 0.01:
            print(f"  {key}: {value:.4e}")
        else:
            print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## 8. Visualization Summary

Four-panel summary of chlorophyll fluorescence characteristics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Emission line shape
ax1 = axes[0, 0]
wave_em = np.linspace(600, 850, 200)
h_single = chl_fl.emission_line_single_gaussian(wave_em)
h_double = chl_fl.emission_line_double_gaussian(wave_em)
ax1.plot(wave_em, h_single, 'b-', linewidth=2, label='Single Gaussian')
ax1.plot(wave_em, h_double, 'r--', linewidth=2, label='Double Gaussian')
ax1.axvline(x=685, color='gray', linestyle=':', alpha=0.7)
ax1.axvline(x=730, color='gray', linestyle=':', alpha=0.7)
ax1.set_xlabel('Emission Wavelength (nm)')
ax1.set_ylabel('h$_C$(λ) (nm$^{-1}$)')
ax1.set_title('Emission Line Shape')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Quantum yield vs PAR
ax2 = axes[0, 1]
PAR = np.linspace(0, 1500, 100)
phi = chl_fl.quantum_yield_irradiance_dependent(PAR)
ax2.plot(PAR, phi * 100, 'g-', linewidth=2)
ax2.axhline(y=2, color='r', linestyle='--', alpha=0.7, label='Default = 2%')
ax2.set_xlabel('PAR (μmol photons m$^{-2}$ s$^{-1}$)')
ax2.set_ylabel('Quantum Yield Φ$_C$ (%)')
ax2.set_title('Quantum Yield vs Irradiance')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel 3: Backscatter coefficients comparison
ax3 = axes[1, 0]
wave_ex = np.linspace(400, 550, 100)
a_ph_spec = 0.03 * np.exp(-((wave_ex - 440) / 40)**2)
bb_fl = chl_fl.fluorescence_backscattering_coeff(a_ph_spec, phi_C=0.02)
bb_raman = raman.raman_backscattering_coeff(wave_ex)
ax3.semilogy(wave_ex, bb_fl * 1e4, 'g-', linewidth=2, label='Fluorescence b$_{bC}$')
ax3.semilogy(wave_ex, bb_raman * 1e4, 'b-', linewidth=2, label='Raman b$_{bR}$')
ax3.set_xlabel('Excitation Wavelength (nm)')
ax3.set_ylabel('b$_b$ (×10$^{-4}$ m$^{-1}$)')
ax3.set_title('Backscatter Coefficients')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Panel 4: FLH schematic
ax4 = axes[1, 1]
wavelengths_flh = np.array([665, 680, 709])
Rrs_example = np.array([0.001, 0.0015, 0.0008])
w = (680 - 665) / (709 - 665)
baseline = 0.001 + w * (0.0008 - 0.001)
ax4.plot(wavelengths_flh, Rrs_example * 1e3, 'bo-', markersize=10, linewidth=2, label='Rrs')
ax4.plot([665, 709], [0.001 * 1e3, 0.0008 * 1e3], 'r--', linewidth=2, label='Baseline')
ax4.plot([680, 680], [baseline * 1e3, 0.0015 * 1e3], 'g-', linewidth=3, label='FLH')
ax4.set_xlabel('Wavelength (nm)')
ax4.set_ylabel('Rrs (×10$^{-3}$ sr$^{-1}$)')
ax4.set_title('Fluorescence Line Height')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(655, 720)

plt.tight_layout()
plt.savefig('chl_fluorescence_summary.png', dpi=150)
plt.show()

print("\nFigure saved to: chl_fluorescence_summary.png")

## 9. Run All Tests

Run the test suite to verify all functions work correctly.

In [ ]:
# Run all tests
passed, failed = test_chl_fl.run_all_tests()

## 10. Individual Test Demonstrations

Here we demonstrate individual tests with verbose output.

In [ ]:
def test_emission_line_normalization():
    """Verify emission line shape is properly normalized."""
    wavelengths = np.linspace(600, 850, 1000)
    
    h_single = chl_fl.emission_line_single_gaussian(wavelengths)
    h_double = chl_fl.emission_line_double_gaussian(wavelengths)
    
    integral_single = np.trapz(h_single, wavelengths)
    integral_double = np.trapz(h_double, wavelengths)
    
    print(f"Single Gaussian integral: {integral_single:.6f}")
    print(f"Double Gaussian integral: {integral_double:.6f}")
    
    assert np.isclose(integral_single, 1.0, rtol=0.01), "Single Gaussian not normalized!"
    assert np.isclose(integral_double, 1.0, rtol=0.01), "Double Gaussian not normalized!"
    print("✓ Both emission lines properly normalized")

test_emission_line_normalization()

In [ ]:
def test_quantum_yield_bounds():
    """Verify quantum yield stays within physical bounds."""
    PAR_values = np.array([0, 10, 100, 500, 1000, 2000, 5000])
    
    phi_values = chl_fl.quantum_yield_irradiance_dependent(PAR_values)
    
    print(f"{'PAR':<10} {'Φ_C':<10}")
    print("-" * 20)
    for par, phi in zip(PAR_values, phi_values):
        print(f"{par:<10} {phi:.4f}")
    
    assert np.all(phi_values >= chl_fl.PHI_FL_HIGH_LIGHT * 0.95), "QY below minimum!"
    assert np.all(phi_values <= chl_fl.PHI_FL_LOW_LIGHT * 1.05), "QY above maximum!"
    assert np.all(np.diff(phi_values) < 0), "QY should decrease with PAR!"
    print("\n✓ Quantum yield within physical bounds")

test_quantum_yield_bounds()

In [ ]:
def test_coefficient_relationships():
    """Verify coefficient relationships: b_bC = 0.5 * b_C = 0.5 * Φ_C * a_ph."""
    a_ph = 0.05
    phi_C = 0.03
    
    b_C = chl_fl.fluorescence_scattering_coeff(a_ph, phi_C)
    bb_C = chl_fl.fluorescence_backscattering_coeff(a_ph, phi_C)
    
    expected_b_C = phi_C * a_ph
    expected_bb_C = 0.5 * phi_C * a_ph
    
    print(f"a_ph = {a_ph} m⁻¹")
    print(f"Φ_C = {phi_C}")
    print(f"")
    print(f"b_C = {b_C:.6f} m⁻¹ (expected: {expected_b_C:.6f})")
    print(f"b_bC = {bb_C:.6f} m⁻¹ (expected: {expected_bb_C:.6f})")
    print(f"b_bC / b_C = {bb_C/b_C:.4f} (expected: 0.5)")
    
    assert np.isclose(b_C, expected_b_C), "b_C incorrect!"
    assert np.isclose(bb_C, expected_bb_C), "b_bC incorrect!"
    assert np.isclose(bb_C / b_C, 0.5), "Backscatter ratio incorrect!"
    print("\n✓ All coefficient relationships verified")

test_coefficient_relationships()

In [ ]:
def test_flh_calculation():
    """Test FLH calculation with known values."""
    # Known values
    Rrs_665 = 0.001
    Rrs_709 = 0.0008
    
    # Linear interpolation to 680 nm
    w = (680 - 665) / (709 - 665)
    baseline = Rrs_665 + w * (Rrs_709 - Rrs_665)
    
    # Case 1: No fluorescence (Rrs at baseline)
    FLH_zero = chl_fl.calc_fluorescence_line_height(Rrs_665, baseline, Rrs_709)
    print(f"Case 1 - No fluorescence:")
    print(f"  Rrs(680) = baseline = {baseline:.6f}")
    print(f"  FLH = {FLH_zero:.2e} (should be ~0)")
    
    # Case 2: With fluorescence
    Rrs_680 = 0.0015
    FLH = chl_fl.calc_fluorescence_line_height(Rrs_665, Rrs_680, Rrs_709)
    expected_FLH = Rrs_680 - baseline
    print(f"\nCase 2 - With fluorescence:")
    print(f"  Rrs(680) = {Rrs_680:.6f}")
    print(f"  Baseline = {baseline:.6f}")
    print(f"  FLH = {FLH:.6f} (expected: {expected_FLH:.6f})")
    
    assert np.isclose(FLH_zero, 0.0, atol=1e-10), "FLH should be zero!"
    assert np.isclose(FLH, expected_FLH), "FLH calculation incorrect!"
    print("\n✓ FLH calculation verified")

test_flh_calculation()

In [ ]:
def test_phase_function_isotropy():
    """Verify fluorescence phase function is isotropic (= 1/4π at all angles)."""
    angles = np.linspace(0, np.pi, 37)  # 0° to 180° in 5° steps
    
    phase_values = chl_fl.fluorescence_phase_function(angles)
    expected = 1.0 / (4 * np.pi)
    
    print(f"Expected value (1/4π): {expected:.6f} sr⁻¹")
    print(f"Min phase function: {phase_values.min():.6f}")
    print(f"Max phase function: {phase_values.max():.6f}")
    print(f"All values equal: {np.allclose(phase_values, expected)}")
    
    assert np.allclose(phase_values, expected), "Phase function not isotropic!"
    print("\n✓ Phase function is isotropic")

test_phase_function_isotropy()